# Buscando Diferencias entre Grupos Pareados

## ANOVA de medidas repetidas con 1 factor

### Procedimiento
- Importar librerias
- Cargar la primera hoja de trabajo de excel en Pandas
- Limpiar los datos
- Prueba de Normalidad (SHAPIRO-WILK)
- Prueba de esfericidad (W de MAUCHLY)
- Prueba ANOVA de medidas repetidas de 1 factor
- (opcional) Para la categoria con 3 o mas grupos HSD Tukey

In [ ]:
#__ Cargar librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import shapiro
from scipy.stats import tukey_hsd
from pingouin import normality
from pingouin import sphericity
from pingouin import rm_anova
from pingouin import pairwise_tukey

In [ ]:
#___ Cargar la hoja de trabajo en un dataframe
anovamr_df = pd.read_excel('datos_analisis_estadistico.xlsx', sheet_name = 'ANOVA 1 MED REP')
anovamr_df

In [ ]:
#___ Transformar la forma del dataframe para utilizar el test de normalidad (SHAPIRO-WILK)
melted_anovamr_df = anovamr_df.melt(id_vars='PACIENTE', value_vars=['ANTES', '1 DIA', '2 DIAS'], var_name='TEMPORALIDAD', value_name='MEDICION')
melted_anovamr_df

In [ ]:
#___ Prueba de normalidad METODO A (RESIDUOS)
promedio_general = melted_anovamr_df['MEDICION'].mean()
promedios_paciente = melted_anovamr_df.groupby('PACIENTE')['MEDICION'].transform('mean')
promedios_tiempo = melted_anovamr_df.groupby('TEMPORALIDAD')['MEDICION'].transform('mean')
residuos = melted_anovamr_df['MEDICION'] - promedios_paciente - promedios_tiempo + promedio_general

# 4. Perform the Shapiro-Wilk test on the residuals
prueba_normalidad_A = shapiro(residuos)
prueba_normalidad_A

In [ ]:
#___ Prueba de normalidad METEDO B (POR GRUPO)
###___ La funcion pg.normality() verifica que una o mas columnas en el dataframe siguen una distribucion normal.
prueba_normalidad = normality(data=melted_anovamr_df, dv='MEDICION', group='TEMPORALIDAD')
prueba_normalidad

In [ ]:
#___ Prueba de normalidad METODO C (VISUAL)
sm.qqplot(residuos, line='s')
plt.title("Grafica Q-Q del Modelo de Residuos")
plt.show()

In [ ]:
#___ Prueba de esfericidad (W de MAUCHLY) FORMATO AMPLIO
##__ ATENCION: Es necesario que el dataframe no contenga la columna de los indices
##__ ya que pingouin accidentalmente incluira los datos de esta columna en los calculos.
prueba_esfe_A = sphericity(anovamr_df.drop('PACIENTE', axis=1))
prueba_esfe_A

In [ ]:
#___ Prueba de esfericidad (W de MAUCHLY) FORMATO LARGO
prueba_esfe_B = sphericity(data=melted_anovamr_df, dv='MEDICION', within='TEMPORALIDAD', subject='PACIENTE')
prueba_esfe_B

In [ ]:
#___ Prueba de ANOVA de medidas repetidas de 1 factor
prueba_anovamr = rm_anova(data=melted_anovamr_df, dv='MEDICION', within='TEMPORALIDAD', subject='PACIENTE')
prueba_anovamr

In [ ]:
#___ Prueba Tukey METODO A (Libreria SciPy)
prueba_tukey_A = tukey_hsd(anovamr_df['ANTES'], anovamr_df['1 DIA'], anovamr_df['2 DIAS'])
print(prueba_tukey_A)


In [ ]:
#___ Prueba Tukey METODO B (Libreria Pingouin)
prueba_tukey_B = pairwise_tukey(data=melted_anovamr_df, dv='MEDICION', between='TEMPORALIDAD')
print(prueba_tukey_B)

In [ ]:
#___ Grafica de error promedio
##___ Prepararr el dataframe
promedio_antes = anovamr_df['ANTES'].mean()
destd_antes = anovamr_df['ANTES'].std()
promedio_dia1 = anovamr_df['1 DIA'].mean()
destd_dia1 = anovamr_df['1 DIA'].std()
promedio_dia2 = anovamr_df['2 DIAS'].mean()
destd_dia2 = anovamr_df['2 DIAS'].std()

plt.figure(figsize=(3,4))
plt.bar(['ANTES', '1 DIA', '2 DIAS'], [promedio_antes, promedio_dia1, promedio_dia2])
plt.errorbar(['ANTES', '1 DIA', '2 DIAS'], [promedio_antes, promedio_dia1, promedio_dia2], yerr=[destd_antes, destd_dia1, destd_dia2], fmt='o', color='r', capsize=10)
plt.ylim(100,150)
plt.xlabel('TEMPORALIDAD')
plt.ylabel('PRESION ARTERIAL (mmHg)')
plt.title('PRESION ARTERIAL')